## Setup

In [1]:
from google.colab import drive
mount = drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# import necessary libraries
import re
import random
import json
from pathlib import Path
import pandas as pd
from PIL import Image



In [3]:
# set seed

SEED = 45
random.seed(SEED)

# define folder paths
DRIVE_DIR = Path('/content/drive/MyDrive/IE7615-FINAL-PROJECT/data')
RAW_DIR = Path("/content/coco/raw")

In [4]:
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# confirm it worked
print(DRIVE_DIR.exists())

True


## Download COCO Dataset

In [6]:
# download coco dataset --> 2017 val Images --> 5k images 1GB

!wget -nc -P /content/coco/raw http://images.cocodataset.org/zips/val2017.zip
!unzip -n -q /content/coco/raw/val2017.zip -d /content/coco/raw
!wget -nc -P /content/coco/raw http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -n -q /content/coco/raw/annotations_trainval2017.zip -d /content/coco/raw


--2026-11-05 20:02:07--  http://images.cocodataset.org/zips/val2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.15.255.78, 16.15.158.57, 16.15.165.117, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.15.255.78|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 815585330 (778M) [application/zip]
Saving to: ‘/content/coco/raw/val2017.zip’

val2017.zip         100%[===================>] 777.80M  60.5MB/s    in 16s     

2026-11-05 20:02:24 (49.1 MB/s) - ‘/content/coco/raw/val2017.zip’ saved [815585330/815585330]

--2026-11-05 20:02:33--  http://images.cocodataset.org/annotations/annotations_trainval2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 16.182.101.210, 16.15.207.17, 52.217.89.192, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|16.182.101.210|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 252907541 (241M) [application/zip]
Saving to: ‘/content/coco/

In [7]:
# confirm download

!ls /content/coco/raw
print(len(list((RAW_DIR / "val2017").glob("*.jpg"))))

annotations  annotations_trainval2017.zip  val2017  val2017.zip
5000


## Load captions

In [10]:
# look inside the captions folder

with open(RAW_DIR / "annotations" / "captions_val2017.json") as f:
      cap_json = json.load(f)


print(cap_json.keys())
print(cap_json["images"][0])
print(cap_json["annotations"][0])
print(len(cap_json["annotations"]))
print(len(cap_json["images"]))

dict_keys(['info', 'licenses', 'images', 'annotations'])
{'license': 4, 'file_name': '000000397133.jpg', 'coco_url': 'http://images.cocodataset.org/val2017/000000397133.jpg', 'height': 427, 'width': 640, 'date_captured': '2013-11-14 17:02:52', 'flickr_url': 'http://farm7.staticflickr.com/6116/6255196340_da26cf2c9e_z.jpg', 'id': 397133}
{'image_id': 179765, 'id': 38, 'caption': 'A black Honda motorcycle parked in front of a garage.'}
25014
5000


The image entries have id, file_name, width, height, license, URLs, date_captured.
The annotations entries have image_id and caption. The id here is just the caption’s own ID number.
The link between them--> a caption’s image_id matches an image’s id.
The 25014 is the annotations count.
The 5000 is the image count.

In [11]:
# turn the lists into dataframes for easier cleaning and joining
# create image df
df_images = pd.DataFrame(cap_json["images"])
#only use necessary columns
df_images = df_images[["id", "file_name", "width", "height"]]
# rename id--> image_id
df_images = df_images.rename(columns={"id":"image_id"})

In [12]:
# create the captions df
df_captions = pd.DataFrame(cap_json["annotations"])
# keep only necessary cols
df_captions = df_captions[["image_id","caption"]]



In [13]:
# checks
print(len(df_captions))
df_images.head()


25014


,image_id,file_name,width,height
0,397133,000000397133.jpg,640,427
1,37777,000000037777.jpg,352,230
2,252219,000000252219.jpg,640,428
3,87038,000000087038.jpg,640,480
4,174482,000000174482.jpg,640,388


In [14]:
# how many images have 5 captions, 6 captions, and so on
print(df_captions.groupby("image_id").size().value_counts())
#shows the captions for first image --> confirm the IDs really match
df_captions[df_captions["image_id"] == 397133]

5    4987
6      12
7       1
Name: count, dtype: int64


,image_id,caption
1811,397133,A man is in a kitchen making pizzas.
1812,397133,Man in apron standing on front of oven with pa...
1819,397133,A baker is working in the kitchen rolling dough.
1822,397133,A person standing by a stove in a kitchen.
1842,397133,A table with pies being made and a person stan...


## Clean Captions

In [19]:
# caption cleaning function
# deletes periods, extra spaces at front and end, collapses repeated spaces
# inside the caption into one
def caption_clean(caption):
  caption = caption.strip()
  caption = caption.rstrip(".")
  caption = re.sub(r"\s+", " ", caption )
  return caption

In [20]:
# confirm function works properly
print(caption_clean("  A man   is in a kitchen.  "))

A man is in a kitchen


In [22]:

# apply caption_clean to every caption and store the results in a new column
df_captions["caption_clean"] = df_captions["caption"].map(caption_clean)
# preview rows where cleaning changed the caption, to confirm it worked
df_captions[df_captions["caption"] != df_captions["caption_clean"]].head(10)

,image_id,caption,caption_clean
0,179765,A black Honda motorcycle parked in front of a ...,A black Honda motorcycle parked in front of a ...
2,190236,An office cubicle with four different types of...,An office cubicle with four different types of...
3,331352,A small closed toilet in a cramped space.,A small closed toilet in a cramped space
4,517069,Two women waiting at a bench next to a street.,Two women waiting at a bench next to a street
5,179765,A black Honda motorcycle with a dark burgundy ...,A black Honda motorcycle with a dark burgundy ...
6,331352,A tan toilet and sink combination in a small r...,A tan toilet and sink combination in a small room
7,190236,The home office space seems to be very cluttered.,The home office space seems to be very cluttered
9,517069,A woman sitting on a bench and a woman standin...,A woman sitting on a bench and a woman standin...
11,331352,This is an advanced toilet with a sink and con...,This is an advanced toilet with a sink and con...
12,46378,A cat eating a bird it has caught.,A cat eating a bird it has caught


In [23]:
# remove short captions and duplicates
# shorter captions are not as useful as a longer one

# count words in each caption
df_captions["n_words"] = df_captions["caption_clean"].str.split().str.len()
n_before = len(df_captions)
# filter out captions that have less than 5 words
df_captions= df_captions[df_captions["n_words"] >= 5]

# drop duplicates
df_captions = df_captions.drop_duplicates(subset=["image_id", "caption_clean"])

# number of deleted rows
num_deleted = n_before - len(df_captions)
print(f"{num_deleted} rows deleted")


9 rows deleted


## Filter images

In [24]:
# filter out bad images
# greyscale, tiny images

# check each image's color mode (RGB = color, L = grayscale)
modes = []
for file_name in df_images["file_name"]:
    path = RAW_DIR / "val2017" / file_name
    with Image.open(path) as img:
        modes.append(img.mode)

# store the modes as a new column and count them
df_images["mode"] = modes
df_images["mode"].value_counts()


,count
mode,
RGB,4990
L,10


In [26]:
#calculate each image’s shorter side
# axis =1 takes the min of each row
df_images["min_side"] = df_images[["width", "height"]].min(axis=1)
print((df_images["min_side"] < 256).sum())
#filter out the bad images
df_keep = df_images[(df_images["mode"] == "RGB") & (df_images["min_side"]>=256)]
# print how many kept
len(df_keep)

42


4948

4,948 images kept, 10 grayscale and 42 under 256 px removed.

## Crop and resize

In [28]:
# create output folders
OUT_DIR = Path("/content/coco/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

(OUT_DIR / "images_512").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "images_299").mkdir(parents=True, exist_ok=True)

In [29]:
# center-crop and resize the images--> COCO images are rectangular and come in many sizes
# so we must make each one square first, then resize it

def crop_resize(img, size):
  w, h = img.size
  s = min(w, h)
  left = (w - s) // 2
  top = (h - s) // 2
  img = img.crop((left, top, left + s, top + s))
  img = img.resize((size, size), Image.BICUBIC)
  return img


In [30]:
!ls /content/coco/processed

images_299  images_512


In [31]:
# processes every kept image, one at a time, and saves two square copies of each
for file_name in df_keep["file_name"]:
    path = RAW_DIR / "val2017" / file_name
    with Image.open(path) as img:
        img_512 = crop_resize(img, 512)
        img_512.save(OUT_DIR / "images_512" / file_name, quality=95)
        img_299= crop_resize(img, 299)
        img_299.save(OUT_DIR / "images_299" / file_name, quality=95)


In [32]:
# confirm two new folders, each with 4,948 square images
print(len(list((OUT_DIR / "images_512").glob("*.jpg"))))
print(len(list((OUT_DIR / "images_299").glob("*.jpg"))))

4948
4948


## Build manifest

In [33]:
#build a manifest--> a single table listing each image with its file path and its prompt
# keep only captions for kept images
caps_keep = df_captions[df_captions["image_id"].isin(df_keep["image_id"])]
# pick one prompt per image, at random
prompts = caps_keep.groupby("image_id").sample(1, random_state=SEED)
prompts = prompts.rename(columns={"caption_clean":"prompt"})
prompts = prompts[["image_id", "prompt"]]
manifest = df_keep.merge(prompts, on="image_id")

manifest["path_512"] = "images_512/" + manifest["file_name"]
manifest.to_csv(OUT_DIR / "manifest.csv", index=False)

#checks
print(len(manifest))
manifest.head()

4948


,image_id,file_name,width,height,mode,min_side,prompt,path_512
0,397133,000000397133.jpg,640,427,RGB,427,A person standing by a stove in a kitchen,images_512/000000397133.jpg
1,252219,000000252219.jpg,640,428,RGB,428,a person with a shopping cart on a city street,images_512/000000252219.jpg
2,87038,000000087038.jpg,640,480,RGB,480,a man falls off his skateboard in a skate park,images_512/000000087038.jpg
3,174482,000000174482.jpg,640,388,RGB,388,A blue bicycle sits on a sidewalk near a street,images_512/000000174482.jpg
4,403385,000000403385.jpg,640,511,RGB,511,A bathroom that has a broken wall in the shower,images_512/000000403385.jpg


## Prompt files

Searched the captions by keyword to hand-pick 5 varied test prompts: a person in a kitchen, a motorcycle, a cake, a dog and cat, and a skier.

In [36]:
# make the prompt files
# Test set (5 prompts) --> an animal, a person, food, a vehicle, and an indoor scene
manifest[manifest["prompt"].str.contains("dog", case=False)].head()


,image_id,file_name,width,height,mode,min_side,prompt,path_512
35,219578,000000219578.jpg,640,427,RGB,427,A dog and cat are sleeping together on an oran...,images_512/000000219578.jpg
52,185250,000000185250.jpg,399,640,RGB,399,A man watches as a dog prepares to jump for a ...,images_512/000000185250.jpg
129,494869,000000494869.jpg,427,640,RGB,427,A woman standing at a kitchen counter with a c...,images_512/000000494869.jpg
146,29393,000000029393.jpg,500,375,RGB,375,a brown and white dog a fence and a tree,images_512/000000029393.jpg
173,512836,000000512836.jpg,480,640,RGB,480,"A person with a white umbrella, with two dogs",images_512/000000512836.jpg


In [42]:
manifest[manifest["prompt"].str.contains("person", case=False)].head()



,image_id,file_name,width,height,mode,min_side,prompt,path_512
0,397133,000000397133.jpg,640,427,RGB,427,A person standing by a stove in a kitchen,images_512/000000397133.jpg
1,252219,000000252219.jpg,640,428,RGB,428,a person with a shopping cart on a city street,images_512/000000252219.jpg
14,348881,000000348881.jpg,640,462,RGB,462,A large white airplane and a person on a lot,images_512/000000348881.jpg
64,80671,000000080671.jpg,640,427,RGB,427,A person is trying to ski down a snowy slope,images_512/000000080671.jpg
69,35197,000000035197.jpg,429,640,RGB,429,A black and white photograph of a person weari...,images_512/000000035197.jpg


One interesting finding is the “person” search turned up “A black and white photograph of a person...” (image 35197). Its mode is RGB, so the grayscale filter kept it, even though it’s visually black and white. Some photos must be saved in color format but contain only gray pixels, and checking the mode doesn’t catch them.

In [43]:
manifest[manifest["prompt"].str.contains("cake", case=False)].head()


,image_id,file_name,width,height,mode,min_side,prompt,path_512
81,173383,000000173383.jpg,640,425,RGB,425,a two layered cake sits on a table top,images_512/000000173383.jpg
83,180560,000000180560.jpg,640,426,RGB,426,a person holding a small cake with candles,images_512/000000180560.jpg
86,321214,000000321214.jpg,640,480,RGB,480,The young is enjoying his cake so much its all...,images_512/000000321214.jpg
138,117425,000000117425.jpg,640,426,RGB,426,A toddler celebrates his birthday with a cupcake,images_512/000000117425.jpg
141,504711,000000504711.jpg,640,483,RGB,483,A piece of chocolate cake sits on a plate with...,images_512/000000504711.jpg


In [44]:
manifest[manifest["prompt"].str.contains("honda", case=False)].head()


,image_id,file_name,width,height,mode,min_side,prompt,path_512
3310,179765,000000179765.jpg,640,480,RGB,480,A black Honda motorcycle with a dark burgundy ...,images_512/000000179765.jpg


In [45]:
manifest[manifest["prompt"].str.contains("kitchen", case=False)].head()

,image_id,file_name,width,height,mode,min_side,prompt,path_512
0,397133,000000397133.jpg,640,427,RGB,427,A person standing by a stove in a kitchen,images_512/000000397133.jpg
121,213086,000000213086.jpg,375,500,RGB,375,A man holding open an oven door in a kitchen,images_512/000000213086.jpg
129,494869,000000494869.jpg,427,640,RGB,427,A woman standing at a kitchen counter with a c...,images_512/000000494869.jpg
131,329219,000000329219.jpg,640,427,RGB,427,there is a small puppy on the kitchen floor,images_512/000000329219.jpg
132,34873,000000034873.jpg,640,480,RGB,480,A kitchen with a counter and a table with chairs,images_512/000000034873.jpg


In [48]:
# SAMPLE 5 PROMPT
# select 5 of the images and save in csv
test_ids = [397133, 179765, 173383, 219578, 80671]
test_df = manifest[manifest["image_id"].isin(test_ids)][["image_id", "file_name", "path_512", "prompt"]]
test_df.to_csv(OUT_DIR / "prompts_test.csv", index=False)
test_df

,image_id,file_name,path_512,prompt
0,397133,000000397133.jpg,images_512/000000397133.jpg,A person standing by a stove in a kitchen
35,219578,000000219578.jpg,images_512/000000219578.jpg,A dog and cat are sleeping together on an oran...
64,80671,000000080671.jpg,images_512/000000080671.jpg,A person is trying to ski down a snowy slope
81,173383,000000173383.jpg,images_512/000000173383.jpg,a two layered cake sits on a table top
3310,179765,000000179765.jpg,images_512/000000179765.jpg,A black Honda motorcycle with a dark burgundy ...


In [49]:
# eval set
eval_df = manifest.sample(1000, random_state=SEED)
eval_df = eval_df[["image_id", "file_name", "path_512", "prompt"]]

eval_df = eval_df.reset_index(drop=True)
eval_df["in_sweep_subset"] = eval_df.index < 500
# save
eval_df.to_csv(OUT_DIR / "prompts_eval.csv", index=False)
# check
print(len(eval_df))
print(eval_df["in_sweep_subset"].sum())

1000
500


## Stats

In [50]:
# collect dataset stats for the proposal and save them as a JSON file
stats = {
    "dataset": "COCO 2017 val",
    "images_raw": int(len(df_images)),
    "captions_raw": int(len(cap_json["annotations"])),
    "captions_removed_short_or_duplicate": int(num_deleted),
    "images_removed_grayscale": int((df_images["mode"] != "RGB").sum()),
    "images_removed_too_small": int(((df_images["mode"] == "RGB") & (df_images["min_side"] < 256)).sum()),
    "images_kept": int(len(manifest)),
    "captions_kept": int(len(caps_keep)),
    "caption_words_mean": round(float(caps_keep["n_words"].mean()), 1),
    "image_size_generation": "512x512 (center crop)",
    "image_size_fid": "299x299 (center crop)",
    "prompts_test": int(len(test_df)),
    "prompts_eval": int(len(eval_df)),
}

with open(OUT_DIR / "dataset_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print(json.dumps(stats, indent=2))



{
  "dataset": "COCO 2017 val",
  "images_raw": 5000,
  "captions_raw": 25014,
  "captions_removed_short_or_duplicate": 9,
  "images_removed_grayscale": 10,
  "images_removed_too_small": 42,
  "images_kept": 4948,
  "captions_kept": 24745,
  "caption_words_mean": 10.4,
  "image_size_generation": "512x512 (center crop)",
  "image_size_fid": "299x299 (center crop)",
  "prompts_test": 5,
  "prompts_eval": 1000
}


## Save to drive

In [53]:
import shutil

# small files--> copy directly
shutil.copy(OUT_DIR / "manifest.csv", DRIVE_DIR / "manifest.csv")
shutil.copy(OUT_DIR / "prompts_test.csv", DRIVE_DIR / "prompts_test.csv")
shutil.copy(OUT_DIR / "prompts_eval.csv", DRIVE_DIR / "prompts_eval.csv")
shutil.copy(OUT_DIR / "dataset_stats.json", DRIVE_DIR / "dataset_stats.json")

#  zip first then copy the zip
zip_path = shutil.make_archive("/content/coco/coco_processed", "zip", OUT_DIR)
shutil.copy(zip_path, DRIVE_DIR / "coco_processed.zip")

PosixPath('/content/drive/MyDrive/IE7615-FINAL-PROJECT/data/coco_processed.zip')

In [54]:
!ls -lh /content/drive/MyDrive/IE7615-FINAL-PROJECT/data

total 733M
-rw------- 1 root root 732M Nov  6 00:03 coco_processed.zip
-rw------- 1 root root  412 Nov  6 00:03 dataset_stats.json
-rw------- 1 root root 583K Nov  6 00:03 manifest.csv
-rw------- 1 root root 108K Nov  6 00:03 prompts_eval.csv
-rw------- 1 root root  526 Nov  6 00:03 prompts_test.csv
